# Experimento

### Criando o meta conhecimento - Exemplo 4
---

Criação do meta dataset

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openml
from pathlib import Path
from time import time

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import cross_validate, StratifiedKFold, LeaveOneOut
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, f1_score

from pymfe.mfe import MFE

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [5]:
def save_csv(data: pd.DataFrame, name: str = "test") -> bool:
    
    path = Path("data")
    path.mkdir(exist_ok=True)
    
    try:
        data.to_csv(str(path) + f"/{name}.csv", index=True)
        return True
    except Exception as e:
        print("[Erro CSV]: ", e)
        return False

In [6]:
# Obtendo a lista de datasets do OpenML:
all_datasets = openml.datasets.list_datasets(output_format='dataframe')
print(f"Total de datasets no OpenML: {len(all_datasets)}")

# Filtrando os datasets com base nos critérios especificados:

filtered_datasets = all_datasets[
    (all_datasets['NumberOfInstances'] >= 200) &
    (all_datasets['NumberOfInstances'] <= 10000) &
    (all_datasets['NumberOfFeatures'] >= 2) &
    (all_datasets['NumberOfFeatures'] <= 50) &
    (all_datasets['NumberOfClasses'] >= 2) &
    (all_datasets['NumberOfClasses'] <= 6) &
    (all_datasets['NumberOfInstancesWithMissingValues'] < all_datasets['NumberOfInstances'] * 0.3) &
    (all_datasets['MinorityClassSize'] >= 20) &
    (all_datasets['NumberOfNumericFeatures'] == all_datasets['NumberOfFeatures'] - 1) &
    (all_datasets['NumberOfInstances'] * all_datasets['NumberOfFeatures'] <= 60000)
].drop_duplicates(subset='name').reset_index(drop=True)

print(f"Quantidade de datasets após filtro: {len(filtered_datasets)}")

# Removendo datasets da mesma família (baseado no nome):
filtered_datasets['family'] = filtered_datasets['name'].str.extract(r'([a-zA-Z]+)')[0]
final_datasets = filtered_datasets.drop_duplicates(subset='family').reset_index(drop=True) 
# Drop FOREX datasets:
final_datasets = final_datasets[~final_datasets['name'].str.contains('FOREX', case=False)].reset_index(drop=True)
print(f"Quantidade de datasets após remoção de famílias: {len(final_datasets)}")
# print("Datasets selecionados:")
# print(final_datasets[['name', 'NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses']])

Total de datasets no OpenML: 6408
Quantidade de datasets após filtro: 329
Quantidade de datasets após remoção de famílias: 94


In [7]:
# Baixando os datasets filtrados:

datasets = {}   # {nome: {'data': X, 'target': y}}
failed = []

for i, row in final_datasets.iterrows():
    did = int(row['did'])
    name = row['name']

    try:
        ds = openml.datasets.get_dataset(did, download_data=True,
                                         download_qualities=False,
                                         download_features_meta_data=False)
        X, y, _, _ = ds.get_data(dataset_format='dataframe',
                                 target=ds.default_target_attribute)

        unique_name = f"{name}_did{did}" # No caso de haver versões duplicadas

        datasets[unique_name] = {'data': X, 'target': y}
        print(f"[Sucesso]: {i}: {unique_name}  ({X.shape[0]}×{X.shape[1]})")

    except Exception as e:
        failed.append(name)
        print(f"[Erro]: {i}: {name}: {e}")

print("\nDownload concluído!")
print(f"Datasets carregados com sucesso: {len(datasets)}")
print(f"Número de falhas: {len(failed)}")

[Sucesso]: 0: balance-scale_did11  (625×4)
[Sucesso]: 1: breast-w_did15  (699×9)
[Sucesso]: 2: diabetes_did37  (768×8)
[Sucesso]: 3: heart-statlog_did53  (270×13)
[Sucesso]: 4: vehicle_did54  (846×18)
[Sucesso]: 5: ionosphere_did59  (351×34)
[Sucesso]: 6: oil_spill_did311  (937×49)
[Sucesso]: 7: SPECTF_did337  (349×44)
[Sucesso]: 8: prnn_synth_did464  (250×2)
[Sucesso]: 9: rmftsa_sleepdata_did679  (1024×2)
[Sucesso]: 10: fri_c3_1000_25_did715  (1000×25)
[Sucesso]: 11: pwLinear_did721  (200×10)
[Sucesso]: 12: analcatdata_supreme_did728  (4052×7)
[Sucesso]: 13: machine_cpu_did733  (209×6)
[Sucesso]: 14: space_ga_did737  (3107×6)
[Sucesso]: 15: pm10_did750  (500×7)
[Sucesso]: 16: strikes_did770  (625×6)
[Sucesso]: 17: quake_did772  (2178×3)
[Sucesso]: 18: disclosure_x_bias_did774  (662×3)
[Sucesso]: 19: bodyfat_did778  (252×14)
[Sucesso]: 20: delta_ailerons_did803  (7129×5)
[Sucesso]: 21: chscase_vine2_did814  (468×2)
[Sucesso]: 22: chatfield_4_did820  (235×12)
[Sucesso]: 23: stock_did841

In [8]:
# Define classifiers
from sklearn.impute import SimpleImputer

classifiers = {
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'Perceptron': Perceptron(random_state=42, max_iter=1000),
    'MLP': MLPClassifier(random_state=42, max_iter=1000)
}

# Store results
results = []

# Iterate through datasets
for dataset_name, dataset in datasets.items():
    print(f'Processing dataset: {dataset_name}')
    
    # Get data and target
    X = dataset['data']
    y = dataset['target']
    
    # Handle missing values
    imputer = KNNImputer(n_neighbors=3)
    try:
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    except Exception as e:
        print(f'Warning: Imputation failed for {dataset_name} with error: {e}')
        print('Falling back to simple imputation strategy.')
        # Use most frequent strategy for string/categorical data
        imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        # Fit and transform the data
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)


    # Handle categorical features
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Encode target if necessary
    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
    
    # Evaluate each classifier
    for clf_name, clf in classifiers.items():
        print(f'  Evaluating {clf_name}...', end=' ')
        
        # 5-fold cross validation
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_results = cross_validate(clf, X, y, cv=cv, scoring='accuracy', 
                                    return_train_score=False)
        
        # Extract fold accuracies
        fold_accs = cv_results['test_score']
        
        # Create result row
        result_row = {
            'Dataset': dataset_name,
            'Classifier': clf_name,
            'acc_fold1': fold_accs[0],
            'acc_fold2': fold_accs[1],
            'acc_fold3': fold_accs[2],
            'acc_fold4': fold_accs[3],
            'acc_fold5': fold_accs[4],
            'acc_mean': fold_accs.mean(),
            'acc_stddev': fold_accs.std(),
            'train_time': cv_results['fit_time'].sum(),
            'test_time': cv_results['score_time'].sum()
        }
        
        results.append(result_row)
        print('Done')

# Create results DataFrame
performances_df = pd.DataFrame(results)

Processing dataset: balance-scale_did11
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: breast-w_did15
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: diabetes_did37
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: heart-statlog_did53
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evaluating LogisticRegression... Done
  Evaluating Perceptron... Done
  Evaluating MLP... Done
Processing dataset: vehicle_did54
  Evaluating DecisionTree... Done
  Evaluating SVM... Done
  Evaluating KNN... Done
  Evalua

In [9]:
import os
warnings.filterwarnings("ignore")
os.environ['PYTHONWARNINGS'] = 'ignore'

# Extract meta-features
meta_features = []

for dataset_name, dataset in list(datasets.items()):
    print(f'Extracting meta-features from {dataset_name}...', end=' ')
    
    # Get data and target
    X = dataset['data']
    y = dataset['target']

    # Handle categorical features
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Handle missing values
    imputer = KNNImputer(n_neighbors=3)
    try:
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    except Exception as e:
        imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    
    # Encode target if necessary
    if y.dtype == 'object':
        le = LabelEncoder()
        y = le.fit_transform(y)
    else:
        # Convert to numpy array if it's a pandas Series
        y = np.array(y)
    
    # Extract meta-features
    try:
        mfe = MFE(groups=['landmarking', 'general', 'statistical',
                           'model-based', 'info-theory', 'relative',
                            'clustering', 'complexity', 'itemset', 
                            'concept'],
                           summary=['median', 'min', 'max', 'mean', 
                                    'sd', 'quantiles', 'histogram'])
        # Ignore warnings during meta-feature extraction:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mfe.fit(X.values, y)
            ft = mfe.extract()
        # reset warnings filter to default after extraction:
        warnings.resetwarnings()        
        
        # Create result row with dataset name and meta-features
        result_row = {'dataset': dataset_name}
        
        meta_features.append(pd.DataFrame(dict(zip(ft[0], ft[1])), index=[dataset_name]))
        print('Done')
    except Exception as e:
        print(f'Error: {e}')

# Create meta-features DataFrame
meta_features_df = pd.concat(meta_features, ignore_index=False)

Extracting meta-features from balance-scale_did11... Done
Extracting meta-features from breast-w_did15... Done
Extracting meta-features from diabetes_did37... Done
Extracting meta-features from heart-statlog_did53... Done
Extracting meta-features from vehicle_did54... Done
Extracting meta-features from ionosphere_did59... Done
Extracting meta-features from oil_spill_did311... Done
Extracting meta-features from SPECTF_did337... Done
Extracting meta-features from prnn_synth_did464... Done
Extracting meta-features from rmftsa_sleepdata_did679... Done
Extracting meta-features from fri_c3_1000_25_did715... Done
Extracting meta-features from pwLinear_did721... Done
Extracting meta-features from analcatdata_supreme_did728... Done
Extracting meta-features from machine_cpu_did733... Done
Extracting meta-features from space_ga_did737... Done
Extracting meta-features from pm10_did750... Done
Extracting meta-features from strikes_did770... Done
Extracting meta-features from quake_did772... Done
Ex

In [10]:
performances_df2 = performances_df.pivot(index='Dataset', columns='Classifier', values='acc_mean')
performances_df2.columns.name = None
performances_df2 = performances_df2.reset_index()
performances_df2

,Dataset,DecisionTree,KNN,LogisticRegression,MLP,Perceptron,SVM
0,Apple_Stock_Price_Trends_(2014-2023)_did46420,0.353728,0.350561,0.382351,0.351341,0.351368,0.377186
1,CPMP-2015-runtime-classification_did41919,0.430710,0.497125,0.563630,0.563594,0.493513,0.521905
2,CastMetal1_did1447,0.807459,0.859441,0.856597,0.663497,0.862378,0.871608
3,CostaMadre1_did1446,0.824181,0.858136,0.861469,0.636554,0.280904,0.871638
4,Creditability-German-Credit-Data_did46416,0.685000,0.648000,0.749000,0.700000,0.611000,0.711000
...,...,...,...,...,...,...,...
89,wall-robot-navigation_did1525,1.000000,0.984604,0.921376,0.984604,0.834680,0.958577
90,waveform_did47155,0.717500,0.801250,0.847500,0.787500,0.773750,0.832500
91,wdbc_did1510,0.910402,0.935010,0.950815,0.927993,0.880640,0.913895
92,wilt_did40983,0.976856,0.979748,0.967971,0.973136,0.933660,0.946063


In [11]:
meta_dataset = performances_df2.merge(
    right=meta_features_df, 
    left_on='Dataset', 
    right_index=True, 
    how='left'
)

# Reorder columns: 'Dataset' first, then meta-features, then classifiers
meta_cols = meta_features_df.columns.tolist()
classifier_cols = performances_df2.columns.drop('Dataset').tolist()
meta_dataset = meta_dataset[['Dataset'] + meta_cols + classifier_cols]

In [12]:
data = pd.DataFrame(meta_dataset)

if save_csv(data=data, name="meta_features_dataset"):
    print("OK! Csv criado com sucesso!")
else:
    print("Erro ao tentar salvar csv!")

OK! Csv criado com sucesso!


### Meta-modelo (treinamento e teste) 

In [8]:
meta_dataset = pd.read_csv('data/meta_dataset.csv', index_col=0)

In [9]:
classifier_cols = meta_dataset.columns[-6:].tolist()

# Find the classifier with the best (maximum) accuracy for each dataset
meta_dataset['Best'] = meta_dataset[classifier_cols].idxmax(axis=1)

In [10]:
def train_and_evaluate_meta_model(meta_dataset):
    # Create a dictionary to store the reuslts:
    summary_of_predictions = {'Dataset':[], 'Best clf (true)':[], 'Perf of best clf (true)':[],
                           'Best clf (pred)':[], 'Perf of best clf (pred)':[]}
    loo = LeaveOneOut()
    feat_import_perf_fold = []
    y_true = meta_dataset['Best'].values
    y_pred = []

    for train_index, test_index in loo.split(meta_dataset):
        # Split the data into training and test sets
        X = meta_dataset.drop(columns=['Dataset', 'Best'] + classifier_cols) # Drop everything except meta-features
        y = meta_dataset['Best']
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        # Train a simple classifier (e.g., Decision Tree) on the training set
        clf = DecisionTreeClassifier(random_state=42)
        clf.fit(X_train, y_train)
        
        # Predict the best classifier for the test dataset
        y_pred.append(clf.predict(X_test)[0])
        
        # Store results in the summary dictionary
        summary_of_predictions['Dataset'].append(meta_dataset['Dataset'].iloc[test_index].values[0])
        summary_of_predictions['Best clf (true)'].append(y_test.values[0])
        summary_of_predictions['Perf of best clf (true)'].append(meta_dataset.loc[test_index, y_test.values[0]].values[0])
        summary_of_predictions['Best clf (pred)'].append(y_pred[-1])
        summary_of_predictions['Perf of best clf (pred)'].append(meta_dataset.loc[test_index, y_pred[-1]].values[0])


    # Create a DataFrame from the summary of predictions
    summary_df = pd.DataFrame(summary_of_predictions)

    # Calculate meta-model accuracy and F1-score
    meta_model_accuracy = accuracy_score(y_true, y_pred)
    meta_model_f1 = f1_score(y_true, y_pred, average='weighted')
    return summary_df, meta_model_accuracy, meta_model_f1

summary_df, meta_model_accuracy, meta_model_f1 = train_and_evaluate_meta_model(meta_dataset)
print(f'Meta-model Accuracy: {meta_model_accuracy:.2f}')
print(f'Meta-model F1-score: {meta_model_f1:.2f}')

NameError: name 'LeaveOneOut' is not defined

In [64]:
classifier_cols = [c for c in meta_dataset.columns if c in ['DecisionTree', 'SVM', 'KNN',
                                                           'LogisticRegression', 'Perceptron', 'MLP']]
meta_feature_cols = [c for c in meta_dataset.columns if c not in ['Dataset', 'Best'] + classifier_cols]

### Algoritmo Genetico

In [20]:
def evaluate_feature_subset(feature_subset):
    X = meta_dataset[feature_subset]
    y = meta_dataset['Best'].values
    loo = LeaveOneOut()
    y_pred = []
    for train_idx, test_idx in loo.split(X):
        clf = DecisionTreeClassifier(random_state=42)
        clf.fit(X.iloc[train_idx], y[train_idx])
        y_pred.append(clf.predict(X.iloc[test_idx])[0])
    return f1_score(y, y_pred, average='weighted')

def random_individual():
    ind = np.random.choice([0, 1], size=len(meta_feature_cols), p=[0.5, 0.5])
    if ind.sum() == 0:
        ind[np.random.randint(len(ind))] = 1
    return ind

def decode_individual(ind):
    return [meta_feature_cols[i] for i, bit in enumerate(ind) if bit == 1]

fitness_cache = {}
def fitness(individual):
    key = tuple(individual.tolist())
    if key not in fitness_cache:
        fitness_cache[key] = evaluate_feature_subset(decode_individual(individual))
    return fitness_cache[key]

def tournament_selection(population, scores, k=3):
    contenders = np.random.choice(len(population), size=k, replace=False)
    best_idx = max(contenders, key=lambda idx: scores[idx])
    return population[best_idx].copy()

def one_point_crossover(parent1, parent2):
    if len(parent1) <= 1:
        return parent1.copy(), parent2.copy()
    point = np.random.randint(1, len(parent1))
    child1 = np.concatenate([parent1[:point], parent2[point:]])
    child2 = np.concatenate([parent2[:point], parent1[point:]])
    return child1, child2

def mutate(individual, mutation_prob=0.05):
    for i in range(len(individual)):
        if np.random.rand() < mutation_prob:
            individual[i] = 1 - individual[i]
    if individual.sum() == 0:
        individual[np.random.randint(len(individual))] = 1

np.random.seed(42)

pop_size = 14
n_generations = 30
crossover_prob = 0.8
mutation_prob = 0.05
elitism = 2

population = [random_individual() for _ in range(pop_size)]
scores = [fitness(ind) for ind in population]

history = []
for generation in range(n_generations):
    ranked = sorted(zip(population, scores), key=lambda x: x[1], reverse=True)
    best_individual, best_score = ranked[0]
    mean_score = np.mean(scores)
    history.append({
        'generation': generation,
        'best_f1': best_score,
        'mean_f1': mean_score,
        'features_count': int(best_individual.sum())
    })
    print(f'Generation {generation}: best F1 = {best_score:.4f}, mean F1 = {mean_score:.4f}, features = {int(best_individual.sum())}')

    new_population = [ind.copy() for ind, _ in ranked[:elitism]]

    while len(new_population) < pop_size:
        parent1 = tournament_selection(population, scores)
        parent2 = tournament_selection(population, scores)
        if np.random.rand() < crossover_prob:
            child1, child2 = one_point_crossover(parent1, parent2)
        else:
            child1, child2 = parent1.copy(), parent2.copy()
        mutate(child1, mutation_prob)
        mutate(child2, mutation_prob)
        new_population.extend([child1, child2])

    population = new_population[:pop_size]
    scores = [fitness(ind) for ind in population]

best_idx = int(np.argmax(scores))
best_individual = population[best_idx]
selected_meta_features_ga = decode_individual(best_individual)
best_meta_model_f1 = scores[best_idx]

print('\nBest feature subset found:')
print(selected_meta_features_ga)
print(f'Best meta-model F1 = {best_meta_model_f1:.4f}')
print(f'Number of selected meta-features = {len(selected_meta_features_ga)}')

history_df = pd.DataFrame(history)
history_df

KeyboardInterrupt: 

In [50]:
list(meta_dataset.columns)[1:len(meta_dataset.columns)-1]

['attr_conc.histogram.0',
 'attr_conc.histogram.1',
 'attr_conc.histogram.2',
 'attr_conc.histogram.3',
 'attr_conc.histogram.4',
 'attr_conc.histogram.5',
 'attr_conc.histogram.6',
 'attr_conc.histogram.7',
 'attr_conc.histogram.8',
 'attr_conc.histogram.9',
 'attr_conc.max',
 'attr_conc.mean',
 'attr_conc.median',
 'attr_conc.min',
 'attr_conc.quantiles',
 'attr_conc.sd',
 'attr_ent.histogram.0',
 'attr_ent.histogram.1',
 'attr_ent.histogram.2',
 'attr_ent.histogram.3',
 'attr_ent.histogram.4',
 'attr_ent.histogram.5',
 'attr_ent.histogram.6',
 'attr_ent.histogram.7',
 'attr_ent.histogram.8',
 'attr_ent.histogram.9',
 'attr_ent.max',
 'attr_ent.mean',
 'attr_ent.median',
 'attr_ent.min',
 'attr_ent.quantiles',
 'attr_ent.sd',
 'attr_to_inst',
 'best_node.histogram.0',
 'best_node.histogram.0.relative',
 'best_node.histogram.1',
 'best_node.histogram.1.relative',
 'best_node.histogram.2',
 'best_node.histogram.2.relative',
 'best_node.histogram.3',
 'best_node.histogram.3.relative',
 

In [47]:
print(meta_dataset['Best'])

0     LogisticRegression
1     LogisticRegression
2                    SVM
3                    SVM
4     LogisticRegression
             ...        
89          DecisionTree
90    LogisticRegression
91    LogisticRegression
92                   KNN
93          DecisionTree
Name: Best, Length: 94, dtype: str


### Modelagem Binaria

In [71]:
import numpy as np
import pandas as pd
from pulp import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import warnings

warnings.filterwarnings("ignore")

def model_mat(max_feature: int = 10, l: float = 0.01):
    # ==========================================
    # 0. DEFINIR A LISTA DE METAFEATURES
    # ==========================================
    # Cole aqui a lista que você enviou

    # ==========================================
    # 1. PREPARAÇÃO E LIMPEZA (Crucial!)
    # ==========================================
    # Garantir que apenas as colunas que realmente existem no DF sejam usadas
    # meta_feature_cols = [c for c in meta_feature_cols if c in meta_dataset.columns]
    # print(meta_feature_cols)
    
    # O PyMFE gera muitos NaNs. Precisamos tratar antes do Random Forest.
    X = meta_dataset[meta_feature_cols]
    y = meta_dataset['Best']

    # ==========================================
    # 2. CALCULAR IMPORTÂNCIA (Random Forest)
    # ==========================================
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X, y)
    feature_scores = rf.feature_importances_
    # print(feature_scores)

    # ==========================================
    # 3. MODELO MILP COM PuLP
    # ==========================================
    problem = LpProblem("MetaFeature_Selection", LpMaximize)

    # Variáveis binárias: x[i] = 1 se a feature i for selecionada
    x = {i: LpVariable(f"{meta_feature_cols[i]}", cat="Binary") for i in range(len(meta_feature_cols))}

    # Hiperparâmetros
    MAX_FEATURES = max_feature  # Número máximo de meta-features que você deseja
    LAMBDA = l     # Penalidade para features com importância muito baixa

    # Função Objetivo: Maximizar importância total menos a penalidade Lambda
    problem += lpSum((feature_scores[i] - LAMBDA) * x[i] for i in range(len(meta_feature_cols)))

    # Restrição: Não selecionar mais que MAX_FEATURES
    problem += lpSum(x[i] for i in range(len(meta_feature_cols))) <= MAX_FEATURES

    # Resolver
    problem.solve(PULP_CBC_CMD(msg=False))

    # ==========================================
    # 4. RESULTADOS
    # ==========================================
    
    # for i in range(len(meta_feature_cols)):
    #     print(value(x[i]))
    #     if value(x[i]) == 1:
    #         print("test")
    
    selected_features = [meta_feature_cols[i] for i in range(len(meta_feature_cols)) if value(x[i]) == 1]
    #print(x)

    # print("-" * 30)
    # print(f"FEATURES SELECIONADAS ({len(selected_features)}):")
    # for f in selected_features:
    #     print(f"- {f}")


    return selected_features


In [74]:
from sklearn.model_selection import cross_validate

selected_features_list = []

for max_feature in range(1, len(meta_feature_cols) + 1):
    result = model_mat(max_feature=max_feature, l=0.002)
    
    if result:
        X = meta_dataset[meta_feature_cols].fillna(0)
        y = meta_dataset['Best']
        X_selected = X[result]
        
        clf = RandomForestClassifier(random_state=42)
        
        # Definimos as métricas que queremos
        metrics = ['f1_weighted', 'accuracy']
        
        # Executa a validação cruzada para ambas as métricas
        cv_results = cross_validate(clf, X_selected, y, cv=5, scoring=metrics)
        
        f1_mean = cv_results['test_f1_weighted'].mean()
        acc_mean = cv_results['test_accuracy'].mean()
        
        # Guardamos em um dicionário para ficar fácil de ler e ordenar
        selected_features_list.append({
            'f1_weighted': f1_mean,
            'accuracy': acc_mean,
            'features': result,
            'n_features': len(result)
        })
        
        print(f"Features: {len(result)} | F1: {f1_mean:.4f} | Acc: {acc_mean:.4f}")

Features: 1 | F1: 0.1973 | Acc: 0.2023
Features: 2 | F1: 0.4329 | Acc: 0.4789
Features: 3 | F1: 0.4817 | Acc: 0.5199
Features: 4 | F1: 0.5008 | Acc: 0.5316
Features: 5 | F1: 0.5292 | Acc: 0.5532
Features: 6 | F1: 0.5371 | Acc: 0.5643
Features: 7 | F1: 0.4888 | Acc: 0.5322
Features: 8 | F1: 0.4813 | Acc: 0.5222
Features: 9 | F1: 0.5214 | Acc: 0.5532
Features: 10 | F1: 0.4862 | Acc: 0.5211
Features: 11 | F1: 0.5122 | Acc: 0.5433
Features: 12 | F1: 0.5281 | Acc: 0.5526
Features: 13 | F1: 0.4906 | Acc: 0.5216
Features: 14 | F1: 0.4743 | Acc: 0.5111
Features: 15 | F1: 0.5149 | Acc: 0.5538
Features: 16 | F1: 0.5069 | Acc: 0.5433
Features: 17 | F1: 0.5289 | Acc: 0.5538
Features: 18 | F1: 0.5420 | Acc: 0.5637
Features: 19 | F1: 0.5018 | Acc: 0.5322
Features: 20 | F1: 0.5345 | Acc: 0.5737
Features: 21 | F1: 0.5060 | Acc: 0.5322
Features: 22 | F1: 0.5468 | Acc: 0.5637
Features: 23 | F1: 0.4983 | Acc: 0.5322
Features: 24 | F1: 0.4905 | Acc: 0.5111
Features: 25 | F1: 0.5184 | Acc: 0.5532
Features:

In [76]:
import json

# Defina o nome do arquivo
nome_arquivo = "data/result_meta_features_choice.json"

# Salvar a lista no arquivo
with open(nome_arquivo, "w", encoding="utf-8") as f:
    json.dump(selected_features_list, f, indent=4, ensure_ascii=False)

print(f"Resultado salvo com sucesso em: {nome_arquivo}")

Resultado salvo com sucesso em: data/result_meta_features_choice.json


In [77]:
# ==========================================
# ORDENAÇÃO E EXIBIÇÃO
# ==========================================

# Ordenar por F1-Weighted (Decrescente)
selected_features_list.sort(key=lambda x: x['f1_weighted'], reverse=True)

print("\n" + "="*30)
print("RANKING FINAL")
print("="*30)

for i, item in enumerate(selected_features_list[:5]):  # Top 5
    print(f"{i+1}º Lugar ({item['n_features']} features):")
    print(f"   - Acurácia: {item['accuracy']:.4f}")
    print(f"   - F1-Score: {item['f1_weighted']:.4f}")


RANKING FINAL
1º Lugar (22 features):
   - Acurácia: 0.5637
   - F1-Score: 0.5468
2º Lugar (37 features):
   - Acurácia: 0.5749
   - F1-Score: 0.5443
3º Lugar (18 features):
   - Acurácia: 0.5637
   - F1-Score: 0.5420
4º Lugar (79 features):
   - Acurácia: 0.5637
   - F1-Score: 0.5373
5º Lugar (6 features):
   - Acurácia: 0.5643
   - F1-Score: 0.5371
